
# Analisis Eksploratif Data Tinggi Air Surabaya
## Exploratory Data Analysis (EDA) untuk `ketinggian_30menit_wide.csv`

Notebook ini dibuat sebagai padanan Surabaya dari `research/dhompo/01_eda_dhompo.ipynb`, tetapi disesuaikan dengan karakter dataset baru:

- data berinterval 30 menit,
- satuan nilai adalah **cm**,
- lokasi utama sementara adalah **Hang Tuah** (`ketinggian_lokasi_1_hang_tuah`),
- banyak sensor pendukung sparse,
- modelling baseline saat ini sangat kuat pada **persistence**.

Tujuan EDA ini bukan hanya visualisasi, tetapi menemukan alasan teknis mengapa model ML sulit mengalahkan persistence dan apa implikasinya untuk pipeline berikutnya.



## Struktur Analisis

1. Inisialisasi environment dan data loader
2. Integritas waktu dan coverage sensor
3. Kanonisasi sinyal A/B dan quality flag
4. Statistik deskriptif dan distribusi target
5. Outlier, reset, dan step jump
6. Pola missingness dan availability sensor
7. Pola temporal harian/mingguan
8. Korelasi dan cross-correlation antar sinyal aktif
9. Autokorelasi target dan kekuatan persistence
10. Kesimpulan dan implikasi modelling


In [ ]:
# Jalankan dari root repository atau folder notebook.
from pathlib import Path
import sys

PROJECT_ROOT = next(
    p for p in (Path.cwd().resolve(), *Path.cwd().resolve().parents)
    if (p / 'pyproject.toml').is_file() and (p / 'src' / 'dhompo').is_dir()
)
for folder in (PROJECT_ROOT, PROJECT_ROOT / 'src', PROJECT_ROOT / 'research'):
    if str(folder) not in sys.path:
        sys.path.insert(0, str(folder))


import warnings
warnings.filterwarnings('ignore')

from pathlib import Path
import sys

sys.path.insert(0, str(PROJECT_ROOT / 'src'))
sys.path.insert(0, str(PROJECT_ROOT))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from dhompo.config import load_yaml_config
from dhompo.data.urban_loader import preprocess_urban_wide_data
from dhompo.data.urban_features import build_urban_forecast_features, build_urban_targets, align_urban_features_targets
from training.evaluate import calc_metrics

try:
    from statsmodels.tsa.stattools import acf
    HAS_STATSMODELS = True
except Exception:
    HAS_STATSMODELS = False

sns.set_theme(style='whitegrid', context='notebook')
plt.rcParams['figure.figsize'] = (14, 5)
plt.rcParams['axes.titlesize'] = 13
plt.rcParams['axes.labelsize'] = 11

CONFIG_PATH = PROJECT_ROOT / 'configs' / 'surabaya' / 'urban_water_level.yaml'
DATA_PATH = PROJECT_ROOT / 'data' / 'surabaya' / 'ketinggian_30menit_wide.csv'
REPORT_PATH = PROJECT_ROOT / 'reports' / 'surabaya' / 'tables' / 'xls_17_surabaya_residual_vs_persistence_ringkas.xlsx'

config = load_yaml_config(CONFIG_PATH)
print(f'Project root: {PROJECT_ROOT}')
print(f'Data path   : {DATA_PATH}')
print(f'Config path : {CONFIG_PATH}')


In [ ]:

data = preprocess_urban_wide_data(config_path=CONFIG_PATH)
raw = data.raw
canonical = data.canonical
modeling = data.modeling_canonical
values = data.values
flags = data.quality_flags
target = data.target_column

print(f'Raw shape       : {raw.shape}')
print(f'Canonical shape : {canonical.shape}')
print(f'Target          : {target}')
print(f'Range           : {raw.index.min()} -> {raw.index.max()}')
print(f'Cadence inferred: {raw.index.to_series().diff().dropna().mode().iloc[0]}')
print(f'Source signals used for current modelling: {data.feature_columns}')



## 1. Integritas Data dan Cadence

Dataset Surabaya sudah berada pada interval 30 menit. Hal ini cocok dengan pipeline Dhompo yang memakai horizon `h1..h5` sebagai `2, 4, 6, 8, 10` step ke depan. Perbedaannya: dataset Surabaya jauh lebih sparse dan nilai target memiliki outlier ekstrem, sehingga tahap cleaning menjadi bagian inti modelling.


In [ ]:

def cadence_report(index: pd.DatetimeIndex) -> pd.DataFrame:
    diffs = index.to_series().diff().dropna().dt.total_seconds().div(60).astype(int)
    return diffs.value_counts().sort_index().rename_axis('Gap Menit').reset_index(name='Count')

summary = pd.DataFrame({
    'Item': ['Rows', 'Columns', 'Start', 'End', 'Target', 'Configured cadence minutes'],
    'Value': [len(raw), raw.shape[1], raw.index.min(), raw.index.max(), target, config.get('cadence_minutes')],
})
display(summary)
display(cadence_report(raw.index))



## 2. Coverage Sensor dan Kanonisasi A/B

Kolom raw memiliki pasangan sensor A/B pada beberapa lokasi. Pipeline Surabaya menggabungkan A/B menjadi satu sinyal kanonis per lokasi dengan aturan:

- jika A dan B tersedia, gunakan rata-rata,
- jika hanya salah satu tersedia, gunakan nilai yang tersedia,
- jika keduanya kosong, hasil kanonis kosong.

Coverage menjadi faktor modelling penting karena sensor dengan coverage sangat rendah cenderung menjadi noise.


In [ ]:

coverage = data.coverage.copy()
coverage['coverage_pct'] = coverage['coverage'] * 100
display(coverage[['column', 'kind', 'location_name', 'non_null', 'total', 'coverage_pct', 'raw_component_non_null']])

fig, ax = plt.subplots(figsize=(12, 4))
plot_cov = coverage.sort_values('coverage_pct', ascending=True)
ax.barh(plot_cov['column'], plot_cov['coverage_pct'], color='#457B9D')
ax.axvline(config.get('features', {}).get('min_coverage', 0.0) * 100, color='#E63946', linestyle='--', label='min_coverage modelling')
ax.set_xlabel('Coverage (%)')
ax.set_title('Coverage Sinyal Kanonis Surabaya')
ax.legend()
plt.tight_layout()
plt.show()



### Interpretasi Coverage

Sinyal aktif untuk modelling saat ini hanya yang melewati threshold coverage. Konsekuensinya, model praktis sangat bergantung pada target Hang Tuah dan sebagian Kalibokor. Sensor Pucang dan PLN Menur terlalu sparse untuk baseline umum, meskipun tetap dipertahankan di statistik deskriptif untuk audit.



## 3. Quality Flag

Pipeline saat ini memakai flag berikut:

- `OK`: nilai observasi asli tersedia.
- `STALE`: nilai asli kosong tetapi masih dapat diisi dari observasi terakhir dalam batas forward-fill terbatas.
- `MISSING`: nilai asli kosong dan tidak ada observasi terakhir yang masih valid.
- `OUTLIER`: khusus target; nilai tersedia tetapi gagal aturan cleaning seperti reset nol, nilai ekstrem, atau lonjakan 30 menit terlalu besar.


In [ ]:

flag_counts = flags.apply(lambda s: s.value_counts()).fillna(0).astype(int).T
flag_counts['Total'] = flag_counts.sum(axis=1)
display(flag_counts)

fig, ax = plt.subplots(figsize=(12, 4))
flag_counts.drop(columns=['Total'], errors='ignore').plot(kind='bar', stacked=True, ax=ax, colormap='tab20')
ax.set_title('Distribusi Quality Flag per Sinyal')
ax.set_ylabel('Jumlah timestamp')
ax.tick_params(axis='x', rotation=45)
plt.tight_layout()
plt.show()



## 4. Statistik Deskriptif

Statistik deskriptif dihitung pada data kanonis asli, sedangkan outlier summary menunjukkan berapa titik target yang dibuang dari label modelling. Nilai tetap ditampilkan dalam **cm**.


In [ ]:

def descriptive_table(df: pd.DataFrame) -> pd.DataFrame:
    rows = []
    for col in df.columns:
        s = df[col].dropna()
        d = s.diff().abs().dropna()
        rows.append({
            'Kolom': col,
            'N': int(s.count()),
            'Mean (cm)': s.mean(),
            'Std (cm)': s.std(),
            'Min (cm)': s.min(),
            'P01 (cm)': s.quantile(0.01),
            'P05 (cm)': s.quantile(0.05),
            'P25 (cm)': s.quantile(0.25),
            'Median (cm)': s.median(),
            'P75 (cm)': s.quantile(0.75),
            'P95 (cm)': s.quantile(0.95),
            'P99 (cm)': s.quantile(0.99),
            'Max (cm)': s.max(),
            'Median |diff30m| (cm)': d.median() if len(d) else np.nan,
            'P99 |diff30m| (cm)': d.quantile(0.99) if len(d) else np.nan,
            'Max |diff30m| (cm)': d.max() if len(d) else np.nan,
        })
    return pd.DataFrame(rows).round(4)

desc = descriptive_table(canonical)
display(desc)
display(data.outlier_summary)


In [ ]:

active_cols = [c for c in data.feature_columns if c in canonical.columns]
fig, axes = plt.subplots(len(active_cols), 2, figsize=(14, 4 * len(active_cols)))
if len(active_cols) == 1:
    axes = np.array([axes])
for i, col in enumerate(active_cols):
    s = canonical[col].dropna()
    sns.histplot(s, bins=80, ax=axes[i, 0], color='#457B9D')
    axes[i, 0].set_title(f'Distribusi {col}')
    axes[i, 0].set_xlabel('cm')
    sns.boxplot(x=s, ax=axes[i, 1], color='#A8DADC')
    axes[i, 1].set_title(f'Boxplot {col}')
    axes[i, 1].set_xlabel('cm')
plt.tight_layout()
plt.show()



## 5. Outlier dan Step Jump Target

Target Hang Tuah memiliki spike dan reset yang cukup besar. Ini adalah alasan utama model level absolut sempat buruk. Dalam pipeline terbaru, titik outlier target tidak dipakai sebagai label training, tetapi tetap dipertahankan di data audit.


In [ ]:

target_raw = canonical[target]
target_clean = modeling[target]
diff = target_raw.diff().abs()

jump_table = pd.DataFrame({
    'Metric': ['count observed', 'count cleaned observed', 'removed outlier', 'diff > 50cm', 'diff > 100cm', 'diff > 200cm', 'max diff 30m'],
    'Value': [
        int(target_raw.notna().sum()),
        int(target_clean.notna().sum()),
        int((target_raw.notna() & target_clean.isna()).sum()),
        int((diff > 50).sum()),
        int((diff > 100).sum()),
        int((diff > 200).sum()),
        float(diff.max()),
    ]
})
display(jump_table)

fig, ax = plt.subplots(figsize=(15, 5))
target_raw.plot(ax=ax, color='#A8DADC', linewidth=1, label='Kanonis asli')
target_clean.plot(ax=ax, color='#1D3557', linewidth=1, label='Setelah cleaning target')
outlier_idx = target_raw[target_raw.notna() & target_clean.isna()].index
ax.scatter(outlier_idx, target_raw.loc[outlier_idx], color='#E63946', s=12, label='OUTLIER target')
ax.set_title('Target Hang Tuah: Data Asli vs Target Modelling')
ax.set_ylabel('Tinggi air (cm)')
ax.legend()
plt.tight_layout()
plt.show()

fig, ax = plt.subplots(figsize=(14, 4))
sns.histplot(diff.dropna().clip(upper=500), bins=80, ax=ax, color='#F4A261')
ax.axvline(100, color='#E63946', linestyle='--', label='threshold cleaning 100 cm/30m')
ax.set_title('Distribusi Perubahan Absolut Target per 30 Menit (clip 500 cm)')
ax.set_xlabel('|diff 30m| (cm)')
ax.legend()
plt.tight_layout()
plt.show()



## 6. Pola Missingness

EDA missingness penting karena model real-time sering menerima data yang tidak lengkap. Visualisasi berikut memperlihatkan kapan sinyal benar-benar tersedia dan kapan pipeline harus mengandalkan forward-fill atau sentinel missing.


In [ ]:

availability = canonical.notna().astype(int)
fig, ax = plt.subplots(figsize=(15, 4))
sns.heatmap(availability.T, cmap=['#F1FAEE', '#1D3557'], cbar=False, ax=ax)
ax.set_title('Availability Sinyal Kanonis Sepanjang Waktu')
ax.set_xlabel('Timestamp index')
ax.set_ylabel('Sinyal')
plt.tight_layout()
plt.show()

monthly_availability = canonical.notna().resample('M').mean().T * 100
display(monthly_availability.round(2))
fig, ax = plt.subplots(figsize=(12, 4))
sns.heatmap(monthly_availability, annot=True, fmt='.1f', cmap='Blues', ax=ax)
ax.set_title('Coverage Bulanan (%)')
plt.tight_layout()
plt.show()



## 7. Pola Temporal Harian dan Mingguan

Pola temporal membantu memeriksa apakah target memiliki siklus operasional/sensor tertentu. Jika tidak ada pola kalender kuat, fitur autoregressive biasanya lebih dominan dibanding fitur jam/hari.


In [ ]:

target_df = pd.DataFrame({'target_cm': modeling[target]}).dropna()
target_df['hour'] = target_df.index.hour + target_df.index.minute / 60
target_df['dayofweek'] = target_df.index.dayofweek

hourly = target_df.groupby('hour')['target_cm'].agg(['mean', 'median', 'std', 'count'])
weekly = target_df.groupby('dayofweek')['target_cm'].agg(['mean', 'median', 'std', 'count'])

display(hourly.round(4).head())
display(weekly.round(4))

fig, axes = plt.subplots(1, 2, figsize=(15, 4))
hourly['median'].plot(ax=axes[0], marker='o', color='#457B9D')
axes[0].set_title('Median Target per Jam')
axes[0].set_xlabel('Jam')
axes[0].set_ylabel('cm')
weekly['median'].plot(ax=axes[1], marker='o', color='#2A9D8F')
axes[1].set_title('Median Target per Hari dalam Minggu')
axes[1].set_xlabel('Day of week (0=Senin)')
axes[1].set_ylabel('cm')
plt.tight_layout()
plt.show()



## 8. Korelasi Antar Sinyal Aktif

Karena hanya sedikit sinyal yang melewati threshold coverage, korelasi dihitung pada sinyal aktif modelling. Korelasi tinggi bukan berarti kausal, tetapi membantu menentukan apakah sensor pendukung benar-benar membawa informasi tambahan terhadap target.


In [ ]:

active_values = modeling[data.feature_columns].copy()
corr = active_values.corr(method='pearson')
display(corr.round(4))

fig, ax = plt.subplots(figsize=(8, 6))
sns.heatmap(corr, annot=True, fmt='.2f', cmap='coolwarm', center=0, ax=ax)
ax.set_title('Korelasi Pearson Sinyal Aktif')
plt.tight_layout()
plt.show()



## 9. Cross-Correlation dan Lag Empiris

Analisis lag mencoba melihat apakah Kalibokor atau curah hujan memiliki hubungan tertunda terhadap Hang Tuah. Karena data sensor pendukung sparse dan tidak selalu sinkron, hasil cross-correlation harus dibaca sebagai indikasi awal, bukan bukti hidrologis final.


In [ ]:

def cross_corr_at_lags(x: pd.Series, y: pd.Series, max_lag: int = 24) -> pd.DataFrame:
    rows = []
    for lag in range(-max_lag, max_lag + 1):
        if lag < 0:
            xs = x.shift(-lag)
        else:
            xs = x.shift(lag)
        aligned = pd.concat([xs, y], axis=1).dropna()
        corr_val = aligned.iloc[:, 0].corr(aligned.iloc[:, 1]) if len(aligned) > 10 else np.nan
        rows.append({'lag_step_30m': lag, 'lag_hours': lag * 0.5, 'corr': corr_val, 'n': len(aligned)})
    return pd.DataFrame(rows)

xcorr_results = {}
for col in data.feature_columns:
    if col == target:
        continue
    xcorr_results[col] = cross_corr_at_lags(modeling[col], modeling[target], max_lag=24)

for col, xcorr_df in xcorr_results.items():
    best = xcorr_df.loc[xcorr_df['corr'].abs().idxmax()]
    print(f'{col}: best lag={best.lag_step_30m:.0f} step ({best.lag_hours:.1f} jam), corr={best.corr:.4f}, n={best.n:.0f}')

fig, ax = plt.subplots(figsize=(13, 5))
for col, xcorr_df in xcorr_results.items():
    ax.plot(xcorr_df['lag_hours'], xcorr_df['corr'], marker='o', label=col)
ax.axhline(0, color='black', linewidth=0.8)
ax.axvline(0, color='black', linewidth=0.8, linestyle='--')
ax.set_title('Cross-Correlation terhadap Target Hang Tuah')
ax.set_xlabel('Lag fitur terhadap target (jam)')
ax.set_ylabel('Pearson correlation')
ax.legend()
plt.tight_layout()
plt.show()



## 10. Autokorelasi Target dan Baseline Persistence

Persistence kuat ketika target sangat berkorelasi dengan nilai terakhirnya. Bagian ini menghitung autokorelasi dan membandingkan baseline persistence sederhana untuk horizon 1 sampai 5 jam.


In [ ]:

feature_cfg = config.get('features', {})
rolling_windows = [(int(w['steps']), str(w['label'])) for w in feature_cfg.get('rolling_windows', [])]
X_features = build_urban_forecast_features(
    values,
    feature_columns=data.feature_columns,
    quality_flags=flags,
    include_quality_flags=bool(feature_cfg.get('include_quality_flags', True)),
    lag_steps=[int(v) for v in feature_cfg.get('lag_steps', [1, 2, 3])],
    rolling_windows=rolling_windows,
)
horizons = [int(h) for h in config.get('horizons', [1, 2, 3, 4, 5])]
y_future = build_urban_targets(modeling, target, horizons)
X_aligned, y_aligned = align_urban_features_targets(X_features, y_future)
current = values[target].loc[X_aligned.index]

rows = []
for h in horizons:
    obs = y_aligned[h]
    pred = current.reindex(obs.index)
    metrics = calc_metrics(obs.to_numpy(), pred.to_numpy())
    rows.append({
        'Horizon': f'+{h} Jam',
        'NSE': metrics['NSE'],
        'RMSE (cm)': metrics['RMSE'],
        'MAE (cm)': metrics['MAE'],
        'PBIAS (%)': metrics['PBIAS'],
        'N': len(obs),
    })
persistence_metrics = pd.DataFrame(rows).round(4)
display(persistence_metrics)

if HAS_STATSMODELS:
    s = modeling[target].dropna()
    acf_vals = acf(s, nlags=48, fft=True)
    fig, ax = plt.subplots(figsize=(13, 4))
    ax.stem(np.arange(len(acf_vals)) * 0.5, acf_vals, basefmt=' ')
    ax.set_title('Autokorelasi Target Hang Tuah sampai 24 Jam')
    ax.set_xlabel('Lag (jam)')
    ax.set_ylabel('ACF')
    plt.tight_layout()
    plt.show()
else:
    print('statsmodels tidak tersedia; ACF dilewati.')



### Sintesis Persistence

Persistence adalah baseline yang memprediksi `level(t+h) = level(t)`. Jika baseline ini sangat kuat, model ML tidak boleh dipaksa mengoreksi semua timestamp. Strategi yang lebih cocok adalah **hybrid**:

- kondisi stabil → pakai persistence,
- kondisi berubah/rising/falling → pakai model residual untuk koreksi.



## 11. Regime Target: Stable, Rising, Falling

Regime awal didefinisikan dari perubahan 1 jam terakhir:

- `rising` jika `level(t) - level(t-1h) > 5 cm`,
- `falling` jika `< -5 cm`,
- sisanya `stable`.

Regime ini dapat dipakai untuk mengevaluasi kapan persistence gagal dan kapan ML residual mungkin berguna.


In [ ]:

regime_df = pd.DataFrame({'target': values[target]})
regime_df['diff_1h'] = regime_df['target'] - regime_df['target'].shift(2)
regime_df['regime'] = np.select(
    [regime_df['diff_1h'] > 5, regime_df['diff_1h'] < -5],
    ['rising', 'falling'],
    default='stable',
)
regime_counts = regime_df.loc[X_aligned.index, 'regime'].value_counts().rename_axis('Regime').reset_index(name='Count')
regime_counts['Percentage'] = regime_counts['Count'] / regime_counts['Count'].sum() * 100
display(regime_counts.round(4))

fig, ax = plt.subplots(figsize=(8, 4))
sns.barplot(data=regime_counts, x='Regime', y='Count', ax=ax, palette='Set2')
ax.set_title('Distribusi Regime berdasarkan Perubahan 1 Jam')
plt.tight_layout()
plt.show()



## 12. Kesimpulan EDA dan Implikasi Modelling

### Temuan Kunci

1. Dataset memiliki cadence 30 menit yang rapi, tetapi coverage sensor tidak merata.
2. Target Hang Tuah adalah sinyal paling lengkap, tetapi memiliki outlier/reset/lonjakan besar.
3. Sensor pendukung utama yang layak dipakai saat ini hanya Kalibokor dan curah hujan Kalibokor.
4. Persistence sangat kuat karena target punya memori/autokorelasi tinggi dan banyak periode stabil.
5. Model ML residual mendekati persistence pada h1, tetapi belum mengalahkannya secara global.

### Implikasi Pipeline

- Gunakan persistence sebagai default operasional.
- ML sebaiknya diposisikan sebagai korektor residual pada regime `rising`/`falling`, bukan prediktor global untuk semua timestamp.
- Cleaning outlier wajib dilakukan sebelum training.
- Evaluasi berikutnya harus per-regime, bukan hanya skor global.
- Untuk horizon 3-5 jam, butuh sinyal eksternal yang lebih informatif atau model tetap persistence sampai coverage sensor pendukung membaik.
